In [1]:
!pip install opendatasets
import opendatasets as od

In [2]:
od.download('https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection/data')

Dataset URL: https://www.kaggle.com/datasets/rmisra/news-headlines-dataset-for-sarcasm-detection


100%|██████████| 3.30M/3.30M [00:00<00:00, 136MB/s]

In [3]:
import numpy as np
import pandas as pd

In [4]:
from datasets import load_dataset, DatasetDict, Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding, AutoModel, AutoConfig
import torch
import torch.nn as nn

In [5]:
path = '/content/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset_v2.json'
df = pd.read_json(path, lines=True)
df.head()

,is_sarcastic,headline,article_link
0,1,thirtysomething scientists unveil doomsday clo...,https://www.theonion.com/thirtysomething-scien...
1,0,dem rep. totally nails why congress is falling...,https://www.huffingtonpost.com/entry/donna-edw...
2,0,eat your veggies: 9 deliciously different recipes,https://www.huffingtonpost.com/entry/eat-your-...
3,1,inclement weather prevents liar from getting t...,https://local.theonion.com/inclement-weather-p...
4,1,mother comes pretty close to using word 'strea...,https://www.theonion.com/mother-comes-pretty-c...


In [6]:
dataset_hf = load_dataset('json', data_files=path)
dataset_hf

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['is_sarcastic', 'headline', 'article_link'],
        num_rows: 28619
    })
})

In [7]:
from transformers.modeling_outputs import TokenClassifierOutput

In [8]:
dataset_hf = dataset_hf.remove_columns(['article_link'])
dataset_hf.set_format('pandas')
dataset_hf = dataset_hf['train'][:]

In [9]:
dataset_hf = dataset_hf.drop_duplicates(subset=['headline'])
dataset_hf = dataset_hf.reset_index()[['headline', 'is_sarcastic']]
dataset_hf = Dataset.from_pandas(dataset_hf)

train_test_valid = dataset_hf.train_test_split(test_size=0.2, seed=15)

test_valid = train_test_valid['test'].train_test_split(test_size=0.5, seed=15)

dataset_hf = DatasetDict({
    'train': train_test_valid['train'],
    'test': test_valid['test'],
    'valid': test_valid['train']
})

dataset_hf

DatasetDict({
    train: Dataset({
        features: ['headline', 'is_sarcastic'],
        num_rows: 22802
    })
    test: Dataset({
        features: ['headline', 'is_sarcastic'],
        num_rows: 2851
    })
    valid: Dataset({
        features: ['headline', 'is_sarcastic'],
        num_rows: 2850
    })
})

In [10]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-cased')
tokenizer.model_max_lenth=512

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [14]:
def tokenize(batch):
  return tokenizer(batch['headline'], truncation=True, max_length=512)

tokenized_dataset = dataset_hf.map(tokenize, batched=True)
tokenized_dataset

Map:   0%|          | 0/22802 [00:00<?, ? examples/s]

Map:   0%|          | 0/2851 [00:00<?, ? examples/s]

Map:   0%|          | 0/2850 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['headline', 'is_sarcastic', 'input_ids', 'attention_mask'],
        num_rows: 22802
    })
    test: Dataset({
        features: ['headline', 'is_sarcastic', 'input_ids', 'attention_mask'],
        num_rows: 2851
    })
    valid: Dataset({
        features: ['headline', 'is_sarcastic', 'input_ids', 'attention_mask'],
        num_rows: 2850
    })
})

In [15]:
tokenized_dataset.set_format('torch', columns=['input_ids', 'is_sarcastic', 'attention_mask'])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
checkpoint = 'distilbert-base-uncased'

In [17]:
class MyCustomModel(nn.Module):
  def __init__(self, checkpoint, num_labels):
    super(MyCustomModel, self).__init__()
    self.num_labels = num_labels
    self.model = model = AutoModel.from_pretrained(checkpoint, config=AutoConfig.from_pretrained(checkpoint,
                                                                                                 output_attention=True,
                                                                                                 output_hidden_states=True))
    self.dropout = nn.Dropout(0.1)
    self.classifier= nn.Linear(768, num_labels)

  def forward(self, input_ids=None, attention_mask=None, labels=None):
    outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
    last_hidden_state = outputs[0]
    sequence_outputs = self.dropout(last_hidden_state)
    logits = self.classifier(sequence_outputs[:, 0, :].view(-1, 768))
    loss = None
    if labels is not None:
      loss_fn = nn.CrossEntropyLoss()
      loss = loss_fn(logits.view(-1, self.num_labels), labels.view(-1))

      return TokenClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions)

In [19]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_scheduler

In [20]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

device(type='cpu')

In [21]:
train_loader = DataLoader(tokenized_dataset['train'], shuffle=True, batch_size=32, collate_fn=data_collator)
valid_loader = DataLoader(tokenized_dataset['valid'], shuffle=True, batch_size=32, collate_fn=data_collator)

Model = MyCustomModel(checkpoint=checkpoint, num_labels=2).to(device)

optim = AdamW(Model.parameters(), lr=5e-5)

num_epochs=3
num_training_steps=num_epochs*len(train_loader)

lr_scheduler = get_scheduler('linear', optimizer=optim, num_warmup_steps=0, num_training_steps=num_training_steps)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [22]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00


In [25]:
from evaluate import load
metric=load('f1')

In [24]:
from tqdm import tqdm

progress_bar_training = tqdm(range(num_training_steps))
progress_bar_eval = tqdm(range(num_epochs*len(valid_loader)))

for epoch in range(num_epochs):
  Model.train()
  for batch in train_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    batch['labels'] = batch.pop('is_sarcastic') # Rename 'is_sarcastic' to 'labels'
    outputs = Model(**batch)
    loss = outputs.loss
    loss.backward()
    optim.step()
    optim.zero_grad()
    progress_bar_training.update(1)

  Model.eval()
  for batch in valid_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    batch['labels'] = batch.pop('is_sarcastic') # Rename 'is_sarcastic' to 'labels'
    with torch.no_grad():
      outputs = Model(**batch)
    logits = outputs.logits
    predictions = np.argmax(logits.cpu(), -1) # Move logits to CPU before converting to numpy
    metric.add_batch(predictions=predictions, references=batch['labels'].cpu()) # Move labels to CPU before adding to metric
    progress_bar_eval.update(1)

  print(metric.compute())

  0%|          | 0/270 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
Model.eval()
test_loader = DataLoader(tokenized_dataset['valid'], shuffle=True, batch_size=32, collate_fn=data_collator)
progress_bar_eval = tqdm(range(num_epochs*len(test_loader)))

for batch in test_loader:
  batch = {k: v.to(device) for k,v in batch.items()}
  batch['labels'] = batch.pop('is_sarcastic')
  with torch.no_grad():
    outputs = Model(**batch)

  logits = outputs.logits
  predictions = np.argmax(logits.cpu(), -1)
  metric.add_batch(predictions=predictions, references=batch['labels'].cpu())
  progress_bar_eval.update(1)

print(metric.compute())



 33%|███▎      | 90/270 [00:12<00:24,  7.22it/s]


  1%|          | 3/270 [00:00<00:11, 24.06it/s]

  2%|▏         | 6/270 [00:00<00:10, 24.38it/s]

  4%|▎         | 10/270 [00:00<00:09, 28.66it/s]

  5%|▌         | 14/270 [00:00<00:08, 31.40it/s]

  7%|▋         | 18/270 [00:00<00:07, 31.55it/s]

  8%|▊         | 22/270 [00:00<00:07, 33.04it/s]

 10%|▉         | 26/270 [00:00<00:07, 33.74it/s]

 11%|█         | 30/270 [00:00<00:07, 33.95it/s]

 13%|█▎        | 34/270 [00:01<00:07, 33.16it/s]

 14%|█▍        | 38/270 [00:01<00:06, 33.62it/s]

 16%|█▌        | 42/270 [00:01<00:06, 33.25it/s]

 17%|█▋        | 46/270 [00:01<00:06, 33.80it/s]

 19%|█▊        | 50/270 [00:01<00:06, 33.84it/s]

 20%|██        | 54/270 [00:01<00:06, 32.40it/s]

 21%|██▏       | 58/270 [00:01<00:06, 32.21it/s]

 23%|██▎       | 62/270 [00:01<00:06, 32.46it/s]

 24%|██▍       | 66/270 [00:02<00:06, 32.73it/s]

 26%|██▌       | 70/270 [00:02<00:06, 32.37it/s]

 27%|██▋       | 74/270 [00:02<00:05, 32.95it/s]


{'f1': 0.8361177406523469}
